In [1]:
import pandas as pd
import os

In [2]:
# Constant map of questions to categories 
ATTENTION_CHECK = 6
REVERSE_CODE = [4, 10, 14, 17]
COGNITIVE = [2, 3, 4, 5, 7]
AFFECTIVE = [8, 9, 10, 11, 12]
EMOTIONAL = [13, 14, 15, 16, 17]
RESONANCE_COLS = ['Positive Resonance_1', 'Positive Resonance_2', 'Positive Resonance_3']

In [3]:
# Get the path prefix - for Jupyter notebooks, use os.getcwd() or os.path.dirname(os.getcwd())
file_prefix = os.path.join(os.path.dirname(os.getcwd()), 'data')
is_reverse_coded = False
filename = 'pilota_data'
df = pd.read_csv(f'{file_prefix}/{filename}.csv')
df = df.iloc[2:]
df = df
df.head()

,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,RecipientLastName,RecipientFirstName,RecipientEmail,ExternalReference,LocationLatitude,...,Positive Resonance_1,Positive Resonance_2,Positive Resonance_3,Age,Race,Race_6_TEXT,Gender,Gender_4_TEXT,Response,Condition
2,100,160,1,2025-10-22 14:39:29,R_5EKFDUivcCIBdFn,NaN,NaN,NaN,NaN,37.423,...,53,31,38,NaN,NaN,NaN,NaN,NaN,It's completely understandable that you're fee...,AI
3,100,136,1,2025-10-22 14:50:46,R_7JD7KkZFtnal4W6,NaN,NaN,NaN,NaN,37.423,...,63,62,20,NaN,NaN,NaN,NaN,NaN,It must have been such a truly joyful and hear...,AI
4,100,69,1,2025-10-22 14:54:22,R_6MdIdImAlbVXcMF,NaN,NaN,NaN,NaN,37.423,...,40,58,39,NaN,NaN,NaN,NaN,NaN,"Oh no, I'm so sorry you fell off your bike – t...",Human
5,100,120,1,2025-10-22 14:57:48,R_6g5EhUpSAFdtu5o,NaN,NaN,NaN,NaN,37.423,...,13,26,9,NaN,NaN,NaN,NaN,NaN,"Oh wow, that sounds like a really disorienting...",AI
6,100,809,1,2025-10-22 15:09:27,R_6hgPU7IhgIiaA4l,NaN,NaN,NaN,NaN,37.7797,...,65,80,50,NaN,NaN,NaN,NaN,NaN,It must have been so lovely to witness such a ...,Human


In [4]:
# Check for duplicate columns and merge emotional_experience columns
def merge_cols_across_conditions(df, column_name):
    # Check if we have both columns
    if column_name in df.columns and f"{column_name}.1" in df.columns:
        df[f'{column_name}'] = df[column_name].fillna(df[f"{column_name}.1"])
        
        # Drop the original duplicate columns
        df = df.drop([f'{column_name}.1'], axis=1)
    return df

def get_column_names(num_range):
    return [f'Empathy_{i}' for i in num_range]

def reverse_code(df, column_name):
    df[column_name] = df[column_name].apply(lambda x: 10 - x)
    return df

columns = ["Emotional_Experience"]
for i in range(1, 18):
    columns.append(f'Empathy_{i}')

for col in columns:
    df = merge_cols_across_conditions(df, col)

# Convert strings to int
for col_name in range(1, 18):
    df[f'Empathy_{col_name}'] = df[f'Empathy_{col_name}'].astype(int)
df[RESONANCE_COLS] = df[RESONANCE_COLS].astype(int)
filtered_df = df
# Filter for Attention Checks 
is_correct = []
for val in df[f'Empathy_{ATTENTION_CHECK}']:
    if val == 10: is_correct.append(1)
    else: is_correct.append(0)
filtered_df['is_correct'] = is_correct

# Reverse Code 
if not is_reverse_coded:
    for col_name in REVERSE_CODE:
        filtered_df = reverse_code(filtered_df, f'Empathy_{col_name}')
    is_reverse_coded = True


# Group by Disaggregated Emotional Category 
cognitive_cols = get_column_names(COGNITIVE)
filtered_df['cognitive'] = filtered_df[cognitive_cols].mean(axis=1)

affective_cols = get_column_names(AFFECTIVE)
filtered_df['affective'] = filtered_df[affective_cols].mean(axis=1)

emotional_cols = get_column_names(EMOTIONAL)
filtered_df['motivational'] = filtered_df[emotional_cols].mean(axis=1)

# Calculate Mean Over Categories 
filtered_df['general_empathy'] = filtered_df[['cognitive', 'affective', 'motivational']].mean(axis=1)

# Calculate Resonance
filtered_df['positive_resonance'] = filtered_df[RESONANCE_COLS].mean(axis=1)

filtered_df.to_csv(f'{file_prefix}/{filename}_filtered.csv', index=False)